# Faixas empíricas de incerteza — Extra Trees final — FD001

Será construída uma faixa simétrica em torno da previsão a partir dos erros OOF, produzidos para motores excluídos do ajuste de cada fold. Dentro de cada fold, metade dos motores será destinada à calibração da largura e a outra metade à avaliação. A separação será feita antes de consultar os resíduos, com semente 42. Cada motor receberá o mesmo peso no cálculo do percentil 90 dos erros absolutos, independentemente do comprimento da trajetória.

A referência de 90% indica o percentil usado na calibração, e não uma cobertura garantida. Como os hiperparâmetros foram escolhidos nesses folds de desenvolvimento, a análise será exploratória. A dependência temporal entre ciclos será preservada pela separação por motor. Será medida tanto a proporção de ciclos cobertos, com peso igual por motor, quanto a proporção de trajetórias inteiramente cobertas; essas duas medidas respondem a perguntas diferentes.

O conjunto oficial de teste não será utilizado. As larguras pertencerão aos modelos OOF de cada fold; não será atribuída ao modelo treinado em todos os dados uma faixa já validada. A implantação de intervalos exige uma calibração dedicada e uma avaliação independente compatível com o uso pretendido.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'cmapss_final_uncertainty.py').exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Projeto não encontrado')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from cmapss_final_uncertainty import evaluate_oof_intervals, interval_summary_by_predicted_rul
REPORT = ROOT / 'outputs' / 'reports'
predictions = pd.read_csv(REPORT / 'extra-trees-oof-erros-fd001.csv')
intervals, widths, engines, assignments = evaluate_oof_intervals(predictions)
display(widths.round(2))
summary = pd.DataFrame([{'motores_avaliacao': len(engines), 
                        'cobertura_media_motor': engines.cobertura.mean(), 
                        'cobertura_trajetoria_inteira': engines.cobertura_trajetoria.mean(), 
                        'largura_media_motor': engines.largura_media.mean(), 
                        'rul_real_abaixo_da_faixa': engines.abaixo.mean(), 
                        'rul_real_acima_da_faixa': engines.acima.mean()}])
display(summary.round(4))

,fold,meia_largura,motores_calibracao,motores_avaliacao,referencia
0,1,38.81,10,10,0.9
1,2,44.88,10,10,0.9
2,3,52.67,10,10,0.9
3,4,46.16,10,10,0.9
4,5,59.85,10,10,0.9


,motores_avaliacao,cobertura_media_motor,cobertura_trajetoria_inteira,largura_media_motor,rul_real_abaixo_da_faixa,rul_real_acima_da_faixa
0,50,0.8587,0.46,91.3362,0.0729,0.0684


Insights: Uma faixa ampla poderá cobrir muitos ciclos e ainda ter pouca utilidade para uma decisão de manutenção. A cobertura média por motor será considerada junto com a largura em ciclos. A cobertura da trajetória inteira será mais exigente, pois basta um ciclo fora da faixa para que aquela trajetória não seja integralmente coberta. Quando o RUL real estiver abaixo do limite inferior, a faixa inteira terá superestimado a vida restante.

In [2]:
bands = interval_summary_by_predicted_rul(intervals)
display(bands.round(3))
order = ['Até 30 ciclos', '31 a 60 ciclos', '61 a 100 ciclos', 'Acima de 100 ciclos']
fig = px.bar(bands.assign(percentual=100*bands.cobertura), x='faixa', y='percentual', category_orders={'faixa': order}, labels={'faixa': 'Faixa de RUL previsto', 'percentual': 'Cobertura média por motor (%)'}, title='Cobertura em motores reservados à avaliação')
fig.update_traces(marker_color='#4477AA')
fig.add_hline(y=90, line_dash='dash', annotation_text='Referência de calibração: 90%')
fig.update_yaxes(range=[0, 105])
fig.show()
px.bar(bands, x='faixa', y='largura_media', category_orders={'faixa': order}, labels={'faixa': 'Faixa de RUL previsto', 'largura_media': 'Largura média (ciclos)'}, title='Largura da faixa por RUL previsto').show()

,faixa,cobertura,largura_media,motores
0,Até 30 ciclos,1.000,63.227,50
1,31 a 60 ciclos,1.000,90.368,50
2,61 a 100 ciclos,0.996,96.951,50
3,Acima de 100 ciclos,0.764,96.951,50


Insights: As faixas do eixo horizontal serão ordenadas pelo RUL previsto, informação disponível ao produzir a previsão. Uma cobertura menor em determinada faixa mostrará uma limitação do intervalo global. A largura poderá variar mesmo com meia largura fixa por fold, pois o limite inferior será truncado em zero e a composição de motores mudará entre categorias.

In [3]:
for name, frame in {'intervalos': intervals, 'larguras': widths, 'motores': engines, 'divisao': assignments, 'faixas': bands, 'resumo': summary}.items():
    frame.to_csv(REPORT / f'extra-trees-incerteza-{name}-fd001.csv', index=False)
text = '# Faixas empíricas de incerteza — Extra Trees final — FD001\n\nPercentil 90 dos erros absolutos, com peso igual por motor. Metade dos motores de cada fold OOF para calibração e metade para avaliação (semente 42). Análise exploratória: folds usados na otimização; sem garantia de cobertura e sem validação de intervalo do modelo refitado.\n\n```text\n' + summary.to_string(index=False) + '\n```\n\n```text\n' + widths.to_string(index=False) + '\n```\n\n```text\n' + bands.to_string(index=False) + '\n```\n'
(REPORT / 'incerteza-extra-trees-final-fd001.md').write_text(text, encoding='utf-8')
print('Tabelas e relatório salvos em:', REPORT)

Tabelas e relatório salvos em: D:\Projetos\CMAPSS_Predictive_Maintenance\outputs\reports


Insights: A análise permitirá identificar se a precisão média esconde grande incerteza em trechos específicos da vida do motor. Estes resultados deverão acompanhar as conclusões do portfólio. A faixa não será utilizada automaticamente para redefinir os alertas escolhidos, pois isso constituiria uma nova política que precisaria ser avaliada.